# 🚀 Starboard AI Agent Demo

This notebook demonstrates how to use the **Starboard AI Agent** programmatically
to analyze and optimize Databricks workloads using **multi-turn conversations**.

## What This Demo Covers
1. **Install** — Install the starboard packages
2. **Configure** — Set up credentials and configuration
3. **Analyze** — Ask the agent to analyze a workload
4. **Follow Up** — Ask contextual follow-up questions in the same session
5. **Inspect** — View execution details and conversation history

## Multi-Turn Conversations
The Starboard SDK maintains conversation context across turns. Ask an initial
question, then follow up naturally — the agent remembers what was previously
discussed, which tools were called, and what entities were discovered.

## Prerequisites
- Databricks workspace with compute attached
- OpenAI API key (or compatible LLM provider)
- Databricks Personal Access Token (PAT) with appropriate permissions

## 1️⃣ Install the Starboard Packages

Install the Starboard packages including the SDK for multi-turn notebook conversations.

In [0]:
%sh 
  ## Install Starboard AI Agent packages
  ## Replace <<GH_PAT>> with a valid GitHub Personal Access Token
  #############################################################
  pip3 install --quiet \
  "starboard-core @ git+https://<<GH_PAT>>@github.com/c-price_data/starboard-ai-agent.git#subdirectory=packages/starboard-core" \
  "starboard @ git+https://<<GH_PAT>>@github.com/c-price_data/starboard-ai-agent.git#subdirectory=packages/starboard"

## 2️⃣ Configure Credentials & Settings

The agent requires:
- **Databricks credentials** - Host URL and Personal Access Token (derived fromm the Notebook context)
- **LLM credentials** - Dervied for Databricks ML Serving Endpoints or use an API key for OpenAI compatible provider

In [0]:
# =============================================================================
# CONFIGURATION
# =============================================================================
import os

# Direct configuration (for demo/development only)
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl", "")
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# LLM Configuration
#LLM_MODEL = "databricks-claude-sonnet-4-5"
LLM_MODEL = "databricks-claude-opus-4-5"

LLM_MAX_TOKENS = 64_000
DATABRICKS_WAREHOUSE_ID="<<WAREHOUSE_ID>>"

# Set environment variables for the CLI
os.environ["DATABRICKS_HOST"] = f"https://{DATABRICKS_HOST}"
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
os.environ["DATABRICKS_WAREHOUSE_ID"] = DATABRICKS_WAREHOUSE_ID

os.environ["LLM_BASE_URL"]=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints"
os.environ["LLM_API_KEY"] = DATABRICKS_TOKEN
os.environ["LLM_MODEL"] = LLM_MODEL
os.environ["LLM_MAX_TOKENS"] = str(LLM_MAX_TOKENS)
os.environ["LLM_TEMPERATURE"] = "0.4"

## 3️⃣ Define Your Conversation

The Starboard Agent supports **multi-turn conversations**. Define your initial
question and any follow-up questions below. The agent maintains full context
between turns — it remembers discovered entities, prior analysis, and tool results.

| Domain | Example Initial Question | Example Follow-Ups |
|--------|--------------------------|---------------------|
| **Query** | "Tune query 1111-2222-3333-4444" | "Would liquid clustering help?" / "What about serverless?" |
| **Job** | "Analyze job 456789 for performance" | "Can we run this as a streaming job?" / "What code changes?" |
| **UC** | "Review table catalog.schema.table" | "Show me the lineage" / "Who has access?" |
| **Warehouse** | "Evaluate warehouse xyz" | "Compare with serverless pricing" |

Edit the goals below to specify your analysis conversation.

In [0]:
INITIAL_QUESTION = "tune job id <<JOB ID>>"
# INITIAL_QUESTION = "tune query id <<QUERY ID>>"

FOLLOW_UP_QUESTIONS = [
    "In terms of cost savings, could we run this as a streaming job instead of hourly?",
    "If yes, what code and config changes would I need to make?",
]

# Session name controls conversation persistence.
# - Set to a fixed name (e.g. "job-266829") to RESUME a previous conversation.
# - Leave as None to always start fresh (a unique name is generated each run).
SESSION_NAME = None

## 4️⃣ Create a Conversation Session

The SDK manages conversation state automatically. Create a session once, then
send as many questions as you like — context carries forward between turns.

In [0]:
import json
import logging
from datetime import datetime
from starboard.infra.observability.logging import setup_structured_logging
from starboard.sdk import StarboardClient

# Suppress debug/info output — change to logging.DEBUG to see full traces
setup_structured_logging(level=logging.WARNING)

client = await StarboardClient.from_env()

# Use a fixed SESSION_NAME above to resume a prior conversation,
# or leave it None to generate a unique name and start fresh each run.
_session_name = SESSION_NAME or f"nb-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
session = await client.create_session(name=_session_name)
print(f"Session: {session.session_name}")

In [ ]:
import time as _time
from starboard.agents.events import (
    ErrorEvent,
    FinalOutputEvent,
    StepCompleteEvent,
    ThinkingEvent,
    ToolEndEvent,
    ToolStartEvent,
    UserInputRequestEvent,
)
from starboard.sdk.models import AgentResponse


async def ask_with_progress(
    session,
    message: str,
    *,
    timeout: float | None = 300.0,
) -> AgentResponse:
    """Send a message and stream real-time progress events (like the CLI).

    Prints tool start/end, errors, and user-input requests as they happen,
    then returns the same ``AgentResponse`` that ``session.ask()`` would.
    """
    final_output = None
    tools_used: list[str] = []
    formatted_report = None
    agent_error = None
    t0 = _time.monotonic()

    async for event in session.ask_stream(message):
        if isinstance(event, ToolStartEvent):
            print(f"  ⏳ {event.friendly_name}…", flush=True)

        elif isinstance(event, ToolEndEvent):
            tools_used.append(event.tool_name)
            icon = "✅" if event.success else "❌"
            print(
                f"  {icon} {event.friendly_name} ({event.duration_seconds:.2f}s)",
                flush=True,
            )

        elif isinstance(event, FinalOutputEvent):
            final_output = (
                event.output
                if isinstance(event.output, dict)
                else event.output.to_dict()
            )

        elif isinstance(event, UserInputRequestEvent):
            print(f"\n  🤔 Agent asks: {event.question}", flush=True)

        elif isinstance(event, ErrorEvent) and not event.is_recoverable:
            agent_error = f"{event.error_type}: {event.error}"
            print(f"  ❌ {agent_error}", flush=True)

    elapsed = _time.monotonic() - t0

    if (
        final_output
        and "complete_report" in final_output
        and final_output["complete_report"]
    ):
        try:
            from starboard.agents.report_formatters import format_agent_report
            formatted_report = format_agent_report(final_output["complete_report"])
        except Exception:
            pass

    return AgentResponse(
        question=message,
        report=formatted_report,
        raw_output=final_output or {},
        tools_used=tools_used,
        tokens_used=final_output.get("tokens_used") if final_output else None,
        cost_usd=final_output.get("cost_usd") if final_output else None,
        duration_seconds=round(elapsed, 2),
        domain=final_output.get("domain") if final_output else None,
        conversation_id=session.session_id,
        turn_number=session.turn_count + 1,
        error=agent_error,
    )

print("ask_with_progress() ready — streams tool events like the CLI.")

In [0]:
print("=" * 70)
print("🚀 Turn 1 — Initial Analysis")
print("=" * 70)
print(f"Question: {INITIAL_QUESTION}\n")

initial_response = await ask_with_progress(session, INITIAL_QUESTION)

if not initial_response.ok:
    print("\n⚠️  Agent error — check details below:")
    print(f"   Error : {initial_response.error}")
    print()
    print("Possible causes:")
    print("  • Databricks output content filtering redacted a numeric ID (e.g. US_BANK_NUMBER)")
    print("    → Try a different job/query ID, or disable workspace PII output filtering.")
    print("  • The job/query ID does not exist in this workspace.")
    print("  • LLM API authentication or quota issue.")
else:
    print()
    print(initial_response.markdown)

## 5️⃣ Follow-Up Questions

Ask follow-up questions in the **same session**. The agent automatically receives
the prior conversation context — discovered entities, tool results, and analysis
from earlier turns inform the response.

In [0]:
follow_up_responses = []

for i, question in enumerate(FOLLOW_UP_QUESTIONS, start=2):
    print("=" * 70)
    print(f"💬 Turn {i} — Follow-Up")
    print("=" * 70)
    print(f"Question: {question}\n")

    response = await ask_with_progress(session, question)
    follow_up_responses.append(response)
    if response.ok:
        print()
        print(response.markdown)
    else:
        print(f"\n⚠️  Error: {response.error}")
    print()

In [0]:
ad_hoc = await ask_with_progress(session, "How can I improve the latency if we convert to streaming?")
print()
print(ad_hoc.markdown)

## 6️⃣ View Structured Output

Each response is an `AgentResponse` with structured fields for programmatic use.

In [0]:
all_responses = [initial_response] + follow_up_responses + [ad_hoc]

print("=" * 60)
print("📊 Conversation Summary")
print("=" * 60)
print(f"Session:  {session.session_name}")
print(f"Turns:    {len(all_responses)}")
print()

for i, resp in enumerate(all_responses, start=1):
    q = resp.question[:60] + ("…" if len(resp.question) > 60 else "")
    tokens = f"{resp.tokens_used:,}" if resp.tokens_used else "n/a"
    cost = f"${resp.cost_usd:.4f}" if resp.cost_usd else "n/a"
    print(f"  Turn {i}: {q}")
    print(f"           domain={resp.domain}  tokens={tokens}  "
          f"cost={cost}  latency={resp.duration_seconds:.1f}s")
    print()

total_tokens = sum(r.tokens_used or 0 for r in all_responses)
total_cost = sum(r.cost_usd or 0.0 for r in all_responses)
print(f"  Total tokens: {total_tokens:,}")
print(f"  Total cost:   ${total_cost:.4f}")
print("=" * 60)

## 7️⃣ Session Management

Sessions persist across notebook restarts. You can list existing sessions,
resume a previous conversation, or start a fresh one.

In [ ]:
sessions = await client.list_sessions()
for s in sessions:
    print(f"  {s.session_name:30s}  turns={s.turn_count}  last={s.updated_at}")

# To resume a specific past session, set SESSION_NAME in the config cell above
# to a name shown here, then re-run from the "Create a Conversation Session" cell.

## 8️⃣ Cleanup

Close the session and release resources when finished.

In [ ]:
await client.close()
print("Client closed — all resources released.")